In [14]:
import wandb
run = wandb.init()

In [16]:
import pytorch_lightning as pl

from torch.nn import functional as F

class LitAutoEncoder(pl.LightningModule):
    def __init__(self, encoder):
        super().__init__()

        self.encoder = encoder

        num_classes = self.encoder.fc.out_features
        print(f"We have {num_classes} classes")


        self.metrics = dict(
            train=torchmetrics.Accuracy(task="multiclass", num_classes=num_classes),
            val=torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        )

    def forward(self, x):
        embedding = self.encoder(x)
        return embedding

    def configure_optimizers(self):

        # optimizer = torch.optim.SGD(self.parameters(), lr=1e-2, momentum=0.9, weight_decay=5e-4)

        # return [optimizer]

        optimizer = torch.optim.Adam(
            self.parameters(), lr=0.0001, weight_decay=0.0
        )

        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=25, gamma=0.5)
        return [optimizer], [scheduler]



    def compute_metric(self, batch, prefix):

        x, y = batch

        logits = self.encoder(x)

        loss = F.cross_entropy(logits, y)

        self.log(f'{prefix}_loss', loss, on_epoch=True)
        self.metrics[prefix].update(logits.detach().cpu(), y.cpu())

        return loss

    def training_step(self, train_batch, batch_idx):


        loss = self.compute_metric(train_batch, 'train')

        return loss

    def summary_metric(self, prefix):

        metric = self.metrics[prefix]
        value = metric.compute()

        self.log(f'{prefix}_acc', value)

        metric.reset()


    def on_train_epoch_end(self):

        self.summary_metric('train')


    def validation_step(self, val_batch, batch_idx):
        x, y = val_batch

        logits = self.encoder(x)

        loss = self.compute_metric(val_batch, 'val')


    def on_validation_epoch_end(self):

        self.summary_metric('val')

In [17]:
from torch import nn 
from torchvision.models.resnet import resnet18

def create_resnet18_with_num_classes(num_classes=100):
    model = resnet18(num_classes=num_classes)
    # why we use this? (See Florian?)
    model.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
    model.maxpool = nn.Identity()

    model.avgpool = nn.AvgPool2d(kernel_size=4)

    return model, "resnet18"

In [18]:
import torchmetrics
import torch

# model = LitAutoEncoder(create_resnet18_with_num_classes()[0])

loading model-lwnx8qeo:v24
We have 100 classes


/home/pat/.cache/pypoetry/virtualenvs/xaikd-bYmevfGI-py3.11/lib/python3.11/site-packages/pytorch_lightning/utilities/migration/utils.py:49: PossibleUserWarning: The loaded checkpoint was produced with Lightning v2.0.6, which is newer than your current Lightning version: v2.0.2
  rank_zero_warn(


LitAutoEncoder(
  (encoder): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): Identity()
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
     

In [21]:
def load_run_id(run_id):

    slug = f"p16i/xaikd-training-teacher-models/{run_id}"
    artifact = run.use_artifact(slug, type='model')
    artifact_dir = artifact.download()
    
    print(f"loading {run_id}")
    model = LitAutoEncoder.load_from_checkpoint(
        f"./artifacts/{run_id}/model.ckpt", map_location=torch.device('cpu'),
        encoder=create_resnet18_with_num_classes()[0]
    )
    model.eval()
    
    print(f"saving {run_id}")
    torch.save(model.encoder.state_dict(), f"./artifacts/{run_id}.pth")

In [22]:
def ano():
    MODEL_MAPPING = {
        "e197": "model-lwnx8qeo:v24",
        "e151": "model-lwnx8qeo:v19",
        "e121": "model-lwnx8qeo:v10",
        "e61":  "model-lwnx8qeo:v9",
        "e23":  "model-lwnx8qeo:v6",
        "e1":   "model-lwnx8qeo:v0"
    }
    
    for k in MODEL_MAPPING.keys():
        
        load_run_id(MODEL_MAPPING[k])
        
ano()

wandb: Downloading large artifact model-lwnx8qeo:v24, 85.70MB. 1 files... 
wandb:   1 of 1 files downloaded.  
Done. 0:0:0.3


loading model-lwnx8qeo:v24
We have 100 classes
saving model-lwnx8qeo:v24


wandb: Downloading large artifact model-lwnx8qeo:v19, 85.70MB. 1 files... 
wandb:   1 of 1 files downloaded.  
Done. 0:0:5.4


loading model-lwnx8qeo:v19
We have 100 classes
saving model-lwnx8qeo:v19


wandb: Downloading large artifact model-lwnx8qeo:v10, 85.70MB. 1 files... 
wandb:   1 of 1 files downloaded.  
Done. 0:0:5.3


...

wandb: Downloading large artifact model-lwnx8qeo:v6, 85.70MB. 1 files... 
wandb:   1 of 1 files downloaded.  
Done. 0:0:3.7


loading model-lwnx8qeo:v6
We have 100 classes
saving model-lwnx8qeo:v6


wandb: Downloading large artifact model-lwnx8qeo:v0, 85.70MB. 1 files... 
wandb:   1 of 1 files downloaded.  
Done. 0:0:4.7


loading model-lwnx8qeo:v0
We have 100 classes
saving model-lwnx8qeo:v0


wandb: Network error (ConnectionError), entering retry loop.


In [24]:
load_run_id("model-vwjm7t8v:v24")

wandb: Downloading large artifact model-vwjm7t8v:v24, 128.53MB. 1 files... 
wandb:   1 of 1 files downloaded.  
Done. 0:0:6.4


loading model-vwjm7t8v:v24
We have 100 classes
saving model-vwjm7t8v:v24


In [7]:

# trainer = pl.Trainer()

# trainer.validate(model, dataloaders=val_dl)

In [8]:
from xaikd.utils import metrics

metrics.accuracy(model, val_dl, num_classes=100, device="cpu")

KeyboardInterrupt: 

In [13]:
print(f"saving {run_id}")
torch.save(model.encoder.state_dict(), f"./artifacts/{run_id}.pth")

saving model-lwnx8qeo:v24
